# LLM answer evaluation (run in Google Colab)

The offline numbers in the README come from the extractive answerer. This notebook runs the same 140 test questions with Claude writing the answer (LangChain + `langchain-anthropic`), keeps the hallucination guard in the loop and saves the metrics.

**Before you start:** add your Anthropic API key in Colab under *Secrets* (key icon on the left) with the name `ANTHROPIC_API_KEY` and allow notebook access. The key is read from the secret store and never printed or written to disk.

Cost: 140 questions with ~1.5k input tokens each on a Haiku-class model, a few cents.

In [ ]:
!git clone -q https://github.com/souadessemyhij1999/engineering-knowledge-copilot.git
%cd engineering-knowledge-copilot
!pip install -q -r requirements.txt pandas

In [ ]:
import os
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
print("key loaded:", bool(os.environ["ANTHROPIC_API_KEY"]))

In [ ]:
# optional: pick another model, e.g. a Sonnet-class model
os.environ["COPILOT_ANSWER_LLM_MODEL"] = "claude-haiku-5-5"
!python scripts/evaluate.py --mode llm | tail -40

In [ ]:
import pandas as pd, json
llm = pd.read_csv("reports/answers_test_llm.csv")
ext = pd.read_csv("reports/answers_test.csv")
m_llm = json.load(open("reports/metrics_llm.json"))
m_ext = json.load(open("reports/metrics.json"))["answers_extractive"]
keys = ["answer_rate (answerable)", "key_fact_accuracy (answered)", "key_fact_accuracy (all answerable)",
        "citation_precision", "guard_pass_rate", "abstention_rate (restricted)", "abstention_rate (unanswerable)",
        "superseded_value_in_answer", "llm_answers_rejected_by_guard", "latency_ms_p50"]
pd.DataFrame({"extractive": [m_ext.get(k) for k in keys], "llm": [m_llm.get(k) for k in keys]}, index=keys)

In [ ]:
# answers where the guard rejected the LLM and the extractive fallback was used
llm[llm.mode_used == "llm->extractive"][["id", "question", "answer"]]

In [ ]:
# zip the results so they can be added to the repo (reports/)
!zip -q llm_results.zip reports/metrics_llm.json reports/answers_test_llm.csv
from google.colab import files
files.download("llm_results.zip")